In [ ]:
from PIL import Image
import requests
from transformers import CLIPProcessor, CLIPModel


In [ ]:
import torch


In [ ]:
class CLIPViTL14FeatureExtractor:
    def __init__(self, device="cuda"):
        self.device = device
        self.model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to(device)
        self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
        self.model.eval()

    def feature_extraction(self, img_path):
        img = Image.open(img_path).convert("RGB")
        inputs = self.processor(images=img, return_tensors="pt").to(self.device)
        with torch.no_grad():
            features = self.model.get_image_features(**inputs)
        return features[0].tolist()      
        
device = "cuda"
extractor = CLIPViTL14FeatureExtractor(device)


In [ ]:
import os
import numpy as np
from tqdm import tqdm

# 1. Đường dẫn dataset trên Kaggle
DATASET_ROOT = "/kaggle/input/vietnamese-foods/Images"   # sửa lại tên dataset của bạn
OUTPUT_ROOT  = "/kaggle/working/ViT-L14-embeddings-vectors"        # nơi lưu vector

os.makedirs(OUTPUT_ROOT, exist_ok=True)

# 3. Hàm xử lý 1 split (Train / Validate / Test)
def process_split(split_name):
    split_input_dir = os.path.join(DATASET_ROOT, split_name)   # vd: /kaggle/input/.../Train
    split_output_dir = os.path.join(OUTPUT_ROOT, split_name)   # vd: /kaggle/working/embeddings/Train

    for class_name in os.listdir(split_input_dir):
        class_input_dir = os.path.join(split_input_dir, class_name)
        if not os.path.isdir(class_input_dir):
            continue

        class_output_dir = os.path.join(split_output_dir, class_name)
        os.makedirs(class_output_dir, exist_ok=True)

        # Lọc các file ảnh
        img_files = [
            f for f in os.listdir(class_input_dir)
            if f.lower().endswith((".png", ".jpg", ".jpeg"))
        ]

        print(f"[{split_name}] Class: {class_name} - {len(img_files)} images")

        for fname in tqdm(img_files):
            img_path = os.path.join(class_input_dir, fname)

            # 1) Trích xuất feature vector (torch tensor [1, 2048])
            feat = extractor.feature_extraction(img_path)

            # 2) Chuyển sang numpy
            feat_np = np.array(feat)   # (2048,)
            
            # 3) Tên file .npy tương ứng
            base_name = os.path.splitext(fname)[0]    # 001
            out_path = os.path.join(class_output_dir, base_name + ".npy")

            # 4) Lưu
            np.save(out_path, feat_np)


In [ ]:
# 4. Chạy cho cả 3 folder
for split in ["Train", "Validate", "Test"]:
    process_split(split)
